## **Notebook Discipline**
- Clear headings for each section so the flow is easy to follow
- Always use "Restart and Run All" before saving to ensure results are reproducible
- No hidden state between runs — every output comes from the visible code cells




### **Importing the required Python packages**


In [6]:
import pandas as pd
from datetime import datetime
import random

### **Configuration and Setup**

In [7]:
# Configuration and Setup

data_path='/content/drive/MyDrive/trendPulse/part1_python_foundations/data'
list_of_countries=['US','GB','IN']
min_views=10000

print(f'Started at: {datetime.now()}')
print(f'countires: {(',').join(list_of_countries)}')
print(f'min_views:{min_views}')


Started at: 2026-10-28 13:25:50.365087
countires: US,GB,IN
min_views:10000


### **Loading function**

In [8]:
def load_country(file_path: str,country,max_rows=None):
  df=pd.read_csv(file_path)
  df['country']=country
  required_columns=['video_id','views', 'likes', 'dislikes', 'comment_count']
  df_columns=df.columns
  for cols in required_columns:
    if cols not in df_columns:
      raise ValueError(f'Missing columns: {cols}')

  records=df.to_dict("records")
  return records

records=[]
for country in list_of_countries:

  path=f'/content/drive/MyDrive/trendPulse/part1_python_foundations/data/{country}videos.csv'
  records.append(load_country(path,country))


category=pd.read_csv('/content/drive/MyDrive/trendPulse/part1_python_foundations/data/categories.csv')
category_names=dict(zip(category['category_id'],category['category_name']))


 ### **Type conversion and cleaning**

In [9]:
# Handle the Type Conversion and Data Cleaning
num_cols=['views','likes','category_id','dislikes','comment_count']

cate_cols=['video_id','title','channel_title','tags','thumbnail_link']
bool_cols=['comments_disabled','ratings_disabled','video_error_or_removed']
failed_count={
    'views':0,
    'likes':0,
    'category_id':0,
    'dislikes':0,
    'comment_count':0,
    'title':0,
    'channel_title':0,
    'tags':0,
    'thumbnail_link':0,
    'comments_disabled':0,
    'ratings_disabled':0,
    'video_error_or_removed':0
}

for country_records in records:
  for record in country_records:
    # Converting the numerical colummns into integer format
    for col in num_cols:
      val=record[col]
      try:
        if str(val).lower() == 'nan' or val is None:
          record[col]=0
        else:
          record[col]=int(record[col])
      except:
        failed_count[col]+=1
        record[col]=0

    # converting the Date and Time into appropriate format
    #1.) Publishing Date
    try:
          record["publish_time"] = datetime.strptime(record["publish_time"], "%Y-%m-%dT%H:%M:%S.%fZ")
    except:
        record["publish_time"] = None
        failed_count["publish_time"]+=1

    #2.) trending_date → compact yy.dd.mm
    try:
      record["trending_date"] = datetime.strptime(record["trending_date"], "%y.%d.%m")
    except:
      record["trending_date"] = None
      failed_count["trending_date"]+=1

    #Converting Categorical Columns into String format
    for col in cate_cols:
      try:
        val=str(record[col])
        if val.lower()==' ' or val.lower()=='nan':
          raise ValueError(f'{col} is Empty')

        else:
          record[col]=str(record[col])
      except:
        record[col]=None
        failed_count[col]+=1
    for col in bool_cols:
      try:
        val=record[col]
        if str(val).strip().lower()=='true':
          record[col]=True
        elif str(val).strip().lower()=='false':
          record[col]=False
        else:
          raise ValueError

      except ValueError:
        record[col]=None
        failed_count[col]+=1









### **Core data structures.**




1.   List of Dictionaries- We are using list of dictionaires because it is easy to store the records, easy to access records and modify the records.
2.   Tuples - We are using tuples  to store the key values such as video id, trending date, view and likes. Tuples are immutable data structure, which is  to ensure we accidently modify these key fields.

1.   Sets - We are using sets here beacuse it ensures we don't have any duplicates values check and use for comparisions like intersection/ differences.  It is mainly used here checking the common between the countries to have a common trending videos.
2.   Dictionaries - We are using Dictionaries here because is easy to store and able to get the values with keys.  Here we are using to have a count of channel apperances with the trending videos and we have us Dictionary in summary to map country -> category -> count.



Mutablity:


*   Mutable: Lists, Dictionaries and Sets
*   Immutable: Tuples
















In [10]:
##Converting the Dictionary into List of Dictonaries
country_records_list=[]
for country_records in records:
  records_list=[]
  for record in country_records:
    records_list.append(record)
  country_records_list.append(records_list)

# extracting the key value as a
key_columns=['video_id', 'trending_date', 'views', 'likes']
record_tuples_list=[]
for records_list in country_records_list:
  for record in records_list:
    record_tuple=()
    record_tuple= tuple(record[key] for key in key_columns)
    record_tuples_list.append(record_tuple)


# Sets for unique video IDs per country

country_unique_ids={}
for country in list_of_countries:
  country_unique_ids[f'{country}_unique_ids']=set()

  for record_list in country_records_list:
    for record in record_list:
      country=record['country']
      video_id=record['video_id']
      country_unique_ids.setdefault(f'{country}_unique_ids',set()).add(video_id)

#Checking the ids in which trending in another country also
us_id=country_unique_ids['US_unique_ids']
gb_id=country_unique_ids['GB_unique_ids']
in_id=country_unique_ids['IN_unique_ids']

us_and_gb=us_id.intersection(gb_id)
us_and_in=us_id.intersection(in_id)
gb_and_in=gb_id.intersection(in_id)


#Checking videos too unique to one country
us_unique_ids=us_id.difference(gb_id)
us_unique_ids=us_unique_ids.difference(in_id)
gb_unique_ids=gb_id.difference(us_id)
gb_unique_ids=gb_unique_ids.difference(in_id)
in_unique_ids=in_id.difference(us_id)
in_unique_ids=in_unique_ids.difference(gb_id)


#Count how many times each channel appears
channel_count={}


for record_list in country_records_list:
  for record in record_list:
    channel=record['channel_title']
    channel_count[channel]=channel_count.get(channel,0)+1

#Top 15 channel with most appearances
top_channels=sorted(channel_count.items(),key=lambda x: x[1],reverse=True)[:15]
channel_position=0
print('Top 15 channels: ')
for channel,count in top_channels:
  channel_position +=1
  print(f'{channel_position}.) channel : {channel}, count: {count}')




#Nested Dictionary Summary
summary={}

for record_list in country_records_list:
  for record in record_list:
    country=record['country']
    record_category_id=record['category_id']
    record_category=category_names[record_category_id]
    no_of_records=0
    summary.setdefault(country,{}).setdefault(record_category,0)
    summary[country][record_category] +=1


for country,categories in summary.items():
  print(f'{country} summary of categories: ')
  print()
  for category, count in list(categories.items()):
    print(f'{category}: {count}')
  print()












Top 15 channels: 
1.) channel : QuietChallenge75, count: 2464
2.) channel : OfficialReview41, count: 1586
3.) channel : UltimateUnboxing65, count: 1433
4.) channel : InsaneReaction46, count: 1233
5.) channel : Brand NewChallenge45, count: 1175
6.) channel : CrazyTrailer91, count: 1018
7.) channel : FinalTrailer17, count: 911
8.) channel : UltimateSpeech20, count: 770
9.) channel : Brand NewSecret51, count: 684
10.) channel : WildHighlights7, count: 682
11.) channel : SurprisingHighlights41, count: 650
12.) channel : FinalBreakdown11, count: 643
13.) channel : CrazyRecipe67, count: 603
14.) channel : QuietInterview9, count: 570
15.) channel : UltimateVlog96, count: 565
US summary of categories: 

People & Blogs: 1423
News & Politics: 755
Music: 3052
Education: 561
Entertainment: 4371
Howto & Style: 713
Comedy: 1366
Gaming: 806
Travel & Events: 262
Sports: 667
Film & Animation: 824
Autos & Vehicles: 404
Pets & Animals: 321
Science & Technology: 555

GB summary of categories: 

Science & 

### **Scope Demonstration**


*   **Local Variable** - The variable which is valid until the function end, after
the function end the variable will invalid
*   **Global Variable** - The variable which is valid until the program end.


*   Below code there is x variable is initialized to 10 outside the function and inside the function fun() variable x is initialized to 5and y variable initialized to 10.  
*   While printing the variables inside the function it prints x=5 and y=10 inside the function the variable x is found so it prints as 5, if the variable x is not found inside the function means only it will check the global variable /outside the function


*   Next printing x varaible and y variable outside the function, x prints as 10 and throws an error because y variable is local variable for the function fun() not a global variable, After the function ends y variable is invalid/ disappeared so it throws and error











In [11]:
x=10

def fun():
  x=5
  y=10
  print(f'Inside the function: x={x}, y={y}')


fun()
print(f'Outside the function: x={x}')

try:
  print(f'Outside the function: y={y}')
except NameError:
  print('variable y cannot be accessible outside the function')


Inside the function: x=5, y=10
Outside the function: x=10
variable y cannot be accessible outside the function


### **Loops over strings and sequences**





*   **For Loop** - For loop  is used to  iterate over the words to count the Uppercase words
*   **For loop with range()** - For loop with range() is used for index access used to access a letters in words to check for Exclamatory marks and Question marks


*   **While Loop** - While loop is used  check the words and if it is emptty means it skip it with the **continue** statement and if the words length exceeds 15 means it will **break**  the loop with break statement.







In [12]:
def title_features(title):
  upper_case_word_count=0
  exclamation_mark_count=0
  question_mark_count=0
  words=title.split()

  for word in words:
    if word.isupper():
      upper_case_word_count +=1

  for i in range(len(title)):
    if title[i]=='!':
      exclamation_mark_count +=1
    elif title[i]=='?':
      question_mark_count +=1
  i=0
  while i < len(words):
    if len(words[i])==0:
      i+=1
      continue
    if len(words[i])> 15:
      break
    i+=1


  return {
      'Upper_case_count' : upper_case_word_count,
      'Exclamation_mark_count' : exclamation_mark_count,
      'Question_mark_count' : question_mark_count,
      'Words Count': len(words)
  }




### **Rule-based classification**

**We classify each video record into one of five labels:**

- **Viral Hit**: views > 1,000,000 and like ratio > 0.9  
- **Steady Performer**: views > 100,000 and like ratio > 0.7  
- **Divisive**: views > 50,000 and like ratio < 0.6  
- **Low Engagement**: views < 50,000  
- **Unrated**: ratings disabled, likes+dislikes = 0, or views = 0  

**Edge cases**:

- If zero view means Unrated
- If Sum of Likes and Dislikes is zero means Unrated
- If rating_disabled means Unrated

**Hand Trace Example**

**Record:**

- views=80,000
- likes=20,000
- dislikes=10,000
- rating_disabled=False

**Steps:**

- views > 50,000
- ratio=20,000/(20,000+10,000)=0.57
- ratio < 0.6
- Meets Divisive condition,  So Label it as 'Divisive'

In [13]:
def classify_video(record):
  views=record.get('views',0)
  likes=record.get('likes',0)
  dislikes=record.get('dislikes',0)
  rating_disabled=record.get('rating_disabled',False)



  if views ==0 or (likes + dislikes == 0):
    return 'Unrated'
  elif rating_disabled==True :
    return 'Unrated'

  ratio=likes/(likes+dislikes)

  if views >=1000000 and ratio >0.9:
    return 'Viral Hit'
  elif views > 100000 and ratio > 0.7:
    return 'Steady Performer'
  elif views > 50000 and ratio < 0.6:
    return 'Divisive'
  elif views < 50000 :
    return 'Low Engagement'
  else:
    return 'Unrated'





label_counts={}

for record_list in country_records_list:
  for record in record_list:
    country=record['country']
    label=classify_video(record)
    label_counts.setdefault(country,{}).setdefault(label,0)
    label_counts[country][label]+=1



for country,labels in label_counts.items():
  print(f'{country} summary of labels: ')
  print()
  for label, count in list(labels.items()):
    print(f'{label}: {count}')
  print()





US summary of labels: 

Low Engagement: 1413
Steady Performer: 9503
Unrated: 2220
Divisive: 71
Viral Hit: 2873

GB summary of labels: 

Unrated: 2009
Steady Performer: 8633
Viral Hit: 3061
Low Engagement: 1325
Divisive: 47

IN summary of labels: 

Steady Performer: 9403
Unrated: 1757
Low Engagement: 875
Divisive: 110
Viral Hit: 2930



### **Refactor**

We can use the int_conversion() helper function to convert the integer safely and not to use this lines repeatedly.  The refactored code  helps the classification function shorter and easier to read and error free.

In [14]:
def int_conversion(record,key,default=0):
  val=record.get(key,default)
  try:
    return int(val)
  except:
    return default